# Lifetime Design Frequencies Workflow

This notebook shows the lifetime design frequencies upload flow as clear notebook steps.

**Conceptual steps**

- Resolve the backend ids for the site, model definition, and turbine locations.
- Load the workbook sheet and detect the frequency columns that end with `[Hz]`.
- Build one prepared row per turbine and reference with site and location metadata attached.
- Create or reuse one analysis for the chosen timestamp.
- Create or update the result rows linked to that analysis.
- Read the saved rows back and rebuild the normalized dataframe.
- Plot the persisted results as comparison, location, and geo views.

**How to run it**

- Run the notebook from top to bottom.
- Set `CREATE_NEW_ANALYSIS = True` to create a new analysis row.
- Set `CREATE_NEW_ANALYSIS = False` to reuse the analysis found for `ANALYSIS_TIMESTAMP`.
- Set `UPLOAD_RESULTS = True` to create or update result rows for turbines with resolved locations.
- Set `UPLOAD_RESULTS = False` to skip writes and only inspect the selected analysis.

## Mermaid Color Legend

All workflow diagrams use the same color meaning.

- <span style="color:#0B5CAD;"><strong>Blue</strong></span>: API call.
- <span style="color:#2E7D32;"><strong>Green</strong></span>: data we keep or check.
- <span style="color:#A56A00;"><strong>Yellow</strong></span>: data we build or reshape.
- <span style="color:#C04A2F;"><strong>Red</strong></span>: choice or stop condition.
- <span style="color:#4B5563;"><strong>Grey line</strong></span>: step-to-step flow.

*Read each diagram from top to bottom.*

## Imports

This section loads the APIs, serializers, workbook tools, and plotting services used later in the notebook.

*Outcome:* all notebook steps use the same local SDK code from this workspace.

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display
from owi.metadatabase.geometry.io import GeometryAPI
from owi.metadatabase.locations.io import LocationsAPI

from owi.metadatabase.results import LifetimeDesignFrequencies, ResultsAPI
from owi.metadatabase.results.models import AnalysisDefinition
from owi.metadatabase.results.serializers import DjangoAnalysisSerializer, DjangoResultSerializer
from owi.metadatabase.results.services import ApiResultsRepository, ResultsService
from owi.metadatabase.results.utils import load_token_from_env_file


## Constants

This section sets the workbook path, API root, project metadata, timestamp, and write controls.

**Check before you run**

- `WORKBOOK` points to the Excel file with the lifetime design frequencies sheet.
- `PROJECTSITE` and `MODEL_DEFINITION` point to the right backend objects.
- `ANALYSIS_TIMESTAMP` picks the analysis row to create or reuse.
- `CREATE_NEW_ANALYSIS` decides whether to create a new analysis.
- `UPLOAD_RESULTS` decides whether result rows are written.

*Outcome:* the notebook has one place to review all runtime choices before any API call.*

<html>
    <style>
        table {
            border: none;
            border-color: transparent;
            border-spacing: 2px;
            border-collapse: separate;
            border-radius: 8px;
            font-family: "Latin Modern Mono";
        }
    </style>
</html>

In [ ]:
WORKSPACE_ROOT = Path.cwd().resolve().parent
DEFAULT_WORKBOOK = WORKSPACE_ROOT / "scripts" / "data" / "results-example-data.xlsx"
DEFAULT_ENV_FILE = WORKSPACE_ROOT / ".env"
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"
DEFAULT_RESULTS_API_ROOT = "https://owimetadatabase-dev.azurewebsites.net/api/v1"
BASE_URL = DEFAULT_RESULTS_API_ROOT
WORKBOOK = DEFAULT_WORKBOOK
PROJECTSITE = "Belwind"
MODEL_DEFINITION = f"as-designed {PROJECTSITE}"
TOKEN = load_token_from_env_file(DEFAULT_ENV_FILE, TOKEN_ENV_VAR)
ANALYSIS_TIMESTAMP = datetime.datetime(2026, 3, 31, 12, 0, 0)
CREATE_NEW_ANALYSIS = True
UPLOAD_RESULTS = True

display(pd.DataFrame([{"workbook": str(WORKBOOK),
                       "projectsite": PROJECTSITE,
                       "api_root": BASE_URL,
                       "token_available": TOKEN is not None,
                       "analysis_timestamp": ANALYSIS_TIMESTAMP,
                       "create_new_analysis": CREATE_NEW_ANALYSIS,
                       "upload_results": UPLOAD_RESULTS}]).T.rename(columns={0: "value"}))
if TOKEN is None:
    raise ValueError(f"Set {TOKEN_ENV_VAR} in {DEFAULT_ENV_FILE} or export "
                     "it in the environment before running this notebook.")


## Resolve Project Metadata

Before reading the workbook, the notebook resolves the backend ids it needs.

**What is resolved**

- `site_id`
- `model_definition_id`
- `location_frame`
- `location_title_to_id_map`
- `existing_analysis_id` for the selected `ANALYSIS_TIMESTAMP`

*Outcome:* later cells can work with stable ids instead of repeating name-based lookups.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Notebook configuration"] --> B["Call Locations API"]
    A --> C["Call Geometry API"]
    A --> D["Call Results API"]
    B --> E["site_id and location_frame"]
    C --> F["model_definition_id"]
    D --> G["existing_analysis_id for the timestamp"]
    E --> H["location_title_to_id_map"]
    F --> I["Metadata ready"]
    G --> I
    H --> I

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B,C,D api;
    class E,F,G,H,I keep;
    class A build;
```

In [ ]:

# ** Class instances for API access and CEIT results management

locations_api = LocationsAPI(api_root=BASE_URL, token=TOKEN)
geometry_api = GeometryAPI(api_root=BASE_URL, token=TOKEN)
results_api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
analysis = LifetimeDesignFrequencies()
analysis_serializer = DjangoAnalysisSerializer()
results_service = ResultsService(repository=ApiResultsRepository(api=results_api))
result_serializer = DjangoResultSerializer()

# -- site_id
site_id = locations_api.get_projectsite_detail(projectsite=PROJECTSITE)["id"]

# -- model_definition_id
model_definition_id = geometry_api.get_modeldefinition_id(
    projectsite=PROJECTSITE, model_definition=MODEL_DEFINITION
)["id"]

# -- location_title_to_id_map
assetlocations = locations_api.get_assetlocations(projectsite=PROJECTSITE)["data"]
location_frame = assetlocations.loc[
    :, [column for column in ["id", "title", "northing", "easting"]
        if column in assetlocations.columns]
].copy()
location_title_to_id_map = {str(row["title"]): int(row["id"])
                            for row in location_frame.to_dict(orient="records")
                            if row.get("title") is not None and row.get("id") is not None}


# -- analysis_id (if existing)
existing_analysis = results_api.get_analysis(
    name=analysis.analysis_name,
    model_definition__id=model_definition_id,
    timestamp=ANALYSIS_TIMESTAMP,
    location__id=None,
)
existing_analysis_id = (
    None if not existing_analysis["exists"] or existing_analysis["id"] is None else int(existing_analysis["id"])
)
# locations_api.plot_assetlocations(projectsite=PROJECTSITE)
display(pd.DataFrame([{"site": PROJECTSITE,
                       "site_id": site_id,
                       "model_definition": MODEL_DEFINITION,
                       "model_definition_id": model_definition_id,
                       "num_locations": len(location_title_to_id_map),
                       "analysis_id": existing_analysis_id}]).T
                     .reset_index().rename(columns={"index": "metric", 0: "value"})
                     .set_index("metric"))


## Load and Inspect the Workbook Data

This stage reads the Excel sheet and identifies the frequency columns that will feed the analysis workflow.

**What happens here**

- Read the `Lifetime -  Design frequencies` sheet from the workbook.
- Build `frequency_frame` as the raw input table.
- Detect the metric columns that end with `[Hz]`.
- Preview the workbook data before analysis and upload logic starts.

*Outcome:* the raw workbook data and metric-column map are ready for analysis creation and row preparation.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Workbook sheet"] --> B["Read Excel data"]
    B --> C["frequency_frame"]
    C --> D["Detect metric columns"]
    D --> E["metric_columns"]
    C --> F["Workbook preview"]
    E --> G["Workbook data ready"]
    F --> G

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B api;
    class C,E,F,G keep;
    class A,D build;
```

In [ ]:

# ** Actual data processing and upload logic starts here --

sheet_name = "Lifetime -  Design frequencies"
frequency_frame = pd.read_excel(WORKBOOK, sheet_name=sheet_name)
metric_columns = {
    str(column).replace(" [Hz]", ""): column
    for column in frequency_frame.columns
    if isinstance(column, str) and column.endswith(" [Hz]")
}
print("\033[95mInput data (frequency frame) loaded from Excel:\033[0m")
display(frequency_frame.head())
print("\033[95mIdentified metric columns in the frequency frame:\033[0m")
display(pd.DataFrame(metric_columns.items(), columns=["metric", "source_column"]))


## Build and Upload the Shared Analysis

This section builds the shared analysis payload, prepares the workbook rows as typed result series, serializes the payloads, and persists them through `ResultsAPI`.

**Execution logic**

- Build the shared `AnalysisDefinition` with `ANALYSIS_TIMESTAMP` so the backend lookup is unique.
- When `CREATE_NEW_ANALYSIS` is `True`, create a new analysis row and fail fast if the timestamped analysis already exists.
- When `CREATE_NEW_ANALYSIS` is `False`, reuse the existing timestamped analysis row.
- Convert the workbook rows into `LifetimeDesignFrequencies` result series with site and location metadata attached.
- Serialize those series into backend payloads with the generic Django serializer.
- When `UPLOAD_RESULTS` is `True`, bulk create or update the result rows for the selected analysis.
- When `UPLOAD_RESULTS` is `False`, skip writes and continue with retrieval against the selected analysis id.

*Outcome:* one shared analysis stores the workbook upload while analysis creation and row upload stay independently controllable.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Prepared workbook data and metadata"] --> B{"Create new analysis?"}
    B -- yes --> C["Create the analysis row"]
    B -- no --> D["Reuse the existing analysis id"]
    C --> E["Selected analysis id"]
    D --> E
    E --> F["Build prepared_rows and result_series"]
    F --> G["Serialize result payloads"]
    G --> H{"Upload result rows?"}
    H -- yes --> I["Bulk create or update location rows"]
    H -- no --> J["Skip writes"]
    I --> K["Upload summary"]
    J --> K

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;
    classDef decision fill:#F7D9D1,stroke:#C04A2F,color:#5A1F14;

    class C,I api;
    class D,E,K keep;
    class A,F,G,J build;
    class B,H decision;
```

In [ ]:

# ** Create analysis (if configured to do so and if not already existing)

analysis_definition = AnalysisDefinition(
    name=analysis.analysis_name,
    model_definition_id=model_definition_id,
    location_id=None,
    source_type="json",
    source=str(WORKBOOK),
    timestamp=ANALYSIS_TIMESTAMP,
    description="Shared CEIT corrosion monitoring upload.",
    additional_data={"input_file": WORKBOOK.name, "sheet_name": sheet_name}
)
analysis_payload = analysis_serializer.to_payload(analysis_definition)

if CREATE_NEW_ANALYSIS and existing_analysis_id is not None:
    raise ValueError(
        "An analysis already exists for the configured name, timestamp, model definition, and location. "
        "Set CREATE_NEW_ANALYSIS to False or choose a different ANALYSIS_TIMESTAMP."
    )

if CREATE_NEW_ANALYSIS:
    created_analysis = results_api.create_analysis(analysis_payload)
    analysis_id = int(created_analysis["id"])
    analysis_created = True
else:
    if existing_analysis_id is None:
        raise ValueError("No existing analysis matched the configured name, timestamp, "
                         "model definition, and location.\nSet CREATE_NEW_ANALYSIS to True "
                         "or adjust ANALYSIS_TIMESTAMP.")
    analysis_id = existing_analysis_id
    analysis_created = False


In [ ]:

# ** Enrich measurements with related metadata and convert to result series for upload

# `result_payloads` are created by first preparing the raw rows from the input DataFrame
# into the structured format expected by the `ResultSeries` model, and then serialized
# into the format expected by the backend API through the `ResultSerializer`.
prepared_rows = [{"turbine": str(row["Turbine"]),
                  "reference": str(row["Reference"]),
                  "site_id": site_id,
                  "location_id": location_title_to_id_map.get(str(row["Turbine"])),
                  **{metric: row.get(source_column)
                     for metric, source_column in metric_columns.items()}}
                 for row in frequency_frame.to_dict(orient="records")]
# analysis.to_results performs the validation step of the prepared rows
# against the result model, and converts them into a series of result
# objects. This step is crucial to ensure that the data conforms to the
# expected structure and types before serialization and upload.
result_series = analysis.to_results({"rows": prepared_rows})


In [ ]:

# ** Upload results to the API (if configured to do so)

results_payloads = [result_serializer.to_payload(series, analysis_id=analysis_id)
                   for series in result_series]
upload_result = {"summary": [], "data": pd.DataFrame(), "exists": False}
if UPLOAD_RESULTS:
    upload_result = results_api.create_or_update_results_bulk(results_payloads)

# --- Sumary

summary_frame = pd.DataFrame(upload_result["summary"],
                             columns=["analysis", "short_description", "result_id", "action"])
payload_summary = pd.DataFrame([{"analysis": int(payload["analysis"]),
                                 "short_description": str(payload["short_description"])}
                                for payload in results_payloads])
upload_summary = payload_summary.merge(summary_frame,
                                       on=["analysis", "short_description"],
                                       how="left")
upload_summary = payload_summary.merge(summary_frame,
                                       on=["analysis", "short_description"],
                                       how="left")
if not upload_summary.empty:
    upload_summary = upload_summary.rename(columns={"short_description": "series_key"})

print("[bold green]Preview of analysis and results payloads[/bold green]")
display(pd.DataFrame([analysis_payload]))
print("[blue]• Sample of results payloads for the first sensor[/blue]")
display(pd.DataFrame(results_payloads[:5]))
print("[blue]• Upload summary[/blue]")
display(upload_summary)
print("[blue]• Summary of uploaded series by sensor and metric[/blue]")
display(pd.DataFrame([{"analysis_id": analysis_id,
                       "analysis_created": analysis_created,
                       "analysis_timestamp": ANALYSIS_TIMESTAMP,
                       "upload_results": UPLOAD_RESULTS,
                       "series_uploaded": len(upload_summary)}]) \
                     .T.rename(columns={0: "value"}))


## Retrieve the Persisted Results

This stage reads the saved rows back from the API and rebuilds the lifetime design dataframe from them.

**What happens here**

- Read the raw result rows for the selected analysis.
- Convert those rows back into result series objects.
- Rebuild the normalized lifetime design dataframe from the saved series.

*Outcome:* the notebook checks the saved backend data, not just the original workbook input.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Selected analysis id"] --> B["Read saved result rows"]
    B --> C["Raw backend table"]
    C --> D["Convert rows back to result series"]
    D --> E["Rebuild the lifetime design dataframe"]
    E --> F["Retrieved dataframe ready"]

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B api;
    class C,F keep;
    class A,D,E build;
```

In [ ]:
# Use the analysis_id obtained above, either from the creation step or from the
# existing analysis query.
raw_results_frame = results_api.list_results(analysis=analysis_id)["data"]
print("\033[95mRaw results frame retrieved from API:\033[0m")
display(raw_results_frame.head())
retrieved_series = [result_serializer.from_mapping(row)
                    for row in raw_results_frame.to_dict(orient="records")]
retrieved_frame = analysis.from_results(retrieved_series)
print("\033[95mRetrieved results frame reconstructed from API data:\033[0m")
display(retrieved_frame.head())


## Plot the Retrieved Results

The final stage renders the lifetime design results from the retrieved backend rows, not from the raw workbook file.

**What to inspect in the plots**

- The comparison plot should group values by reference.
- The location plot should group values by turbine or asset location.
- The geo plot should place the retrieved values on the site coordinates.
- The plotted values should match the rows shown in the retrieved dataframe preview.

*Outcome:* the same persisted analysis can be inspected through the comparison, location, and geo plotting paths exposed by the SDK.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Selected analysis id"] --> B["Create comparison plot"]
    A --> C["Create location plot"]
    A --> D["Create geo plot"]
    B --> E["Comparison widget"]
    C --> F["Location widget"]
    D --> G["Geo widget"]
    E --> H["Display plots"]
    F --> H
    G --> H

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B,C,D api;
    class E,F,G,H keep;
    class A build;
```

In [ ]:
filters = {"analysis_id": analysis_id}

comparison_plot = results_service.plot_results(
    analysis.analysis_name,
    filters=filters,
    plot_type="comparison",
)
location_plot = results_service.plot_results(
    analysis.analysis_name,
    filters=filters,
    plot_type="location",
)
geo_plot = results_service.plot_results(
    analysis.analysis_name,
    filters=filters,
    plot_type="geo",
)

display(comparison_plot.notebook)
display(location_plot.notebook)
display(geo_plot.notebook)


In [ ]:
plot_summary = pd.DataFrame([{"analysis_id": analysis_id,
                              "retrieved_rows": len(raw_results_frame),
                              "normalized_rows": len(retrieved_frame),
                              "series_count": len(retrieved_series),
                              "comparison_plot_available": comparison_plot.notebook is not None,
                              "location_plot_available": location_plot.notebook is not None,
                              "geo_plot_available": geo_plot.notebook is not None}]) \
                            .T.reset_index().rename(columns={"index": "metric", 0: "value"}) \
                            .set_index("metric")

display(plot_summary)
